# Efficient VLM AD Kaggle Runner

Run smoke first to verify the Kaggle environment, HF dataset, feature cache, training, evaluation, and benchmark. After smoke passes on T4 x2, use the full 2-GPU cells to train `RepViT-M1.5 + T5-Efficient-Mini` with Accelerate.

In [ ]:
# Edit these values before running on Kaggle.
GITHUB_REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
GITHUB_BRANCH = "huy"
RUN_FULL_2GPU = False
USE_SAFE_2GPU_CONFIG = False
FULL_DEBUG = True

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["RUN_FULL_2GPU"] = "1" if RUN_FULL_2GPU else "0"
os.environ["USE_SAFE_2GPU_CONFIG"] = "1" if USE_SAFE_2GPU_CONFIG else "0"
os.environ["FULL_DEBUG"] = "1" if FULL_DEBUG else "0"
print({
    "repo": GITHUB_REPO_URL,
    "branch": GITHUB_BRANCH,
    "run_full_2gpu": RUN_FULL_2GPU,
    "use_safe_2gpu_config": USE_SAFE_2GPU_CONFIG,
    "full_debug": FULL_DEBUG,
})


In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
else
  cd Efficient_VLM_For_Autonomous_Driving
  git fetch origin "$GITHUB_BRANCH"
  git checkout "$GITHUB_BRANCH"
  git reset --hard "origin/$GITHUB_BRANCH"
  cd /kaggle/working
fi
cd Efficient_VLM_For_Autonomous_Driving
printf 'branch=' && git rev-parse --abbrev-ref HEAD
printf 'commit=' && git rev-parse HEAD

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
python -m pip install -e .
python -m pip install pycocoevalcap || true

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
assert os.environ["HF_TOKEN"], "HF_TOKEN secret is empty"
print("HF_TOKEN present:", bool(os.environ["HF_TOKEN"]))

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
pwd
git rev-parse --abbrev-ref HEAD
git rev-parse HEAD
nvidia-smi || true
python - <<'PY'
import platform, torch
print('python:', platform.python_version())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda:', torch.version.cuda)
print('cuda device count:', torch.cuda.device_count())
for idx in range(torch.cuda.device_count()):
    print(f'gpu {idx}:', torch.cuda.get_device_name(idx))
PY


## Smoke Debug Run

Each cell is intentionally separate so failures point to the exact pipeline stage.

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad inspect-data --config "$CONFIG" --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset smoke --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad debug-sample --config "$CONFIG" --split train --index 0 --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset smoke --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad train --config "$CONFIG" --stage align --max-steps 20 --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume outputs/repvit_t5_efficient_tiny_smoke/checkpoints/align_latest.pt --max-steps 20 --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint outputs/repvit_t5_efficient_tiny_smoke/checkpoints/finetune_latest.pt --max-samples 32 --debug --debug-samples 3

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG=configs/repvit_t5_efficient_tiny_smoke.yaml
python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint outputs/repvit_t5_efficient_tiny_smoke/checkpoints/finetune_latest.pt --max-samples 32 --debug --debug-samples 3

## Debug Artifacts

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE=outputs/repvit_t5_efficient_tiny_smoke
printf '\nprepared manifest\n'
cat "$PROFILE/prepared_data/manifest.json"
printf '\ncache manifest\n'
cat "$PROFILE/cache/manifest.json"
printf '\npredictions\n'
head -5 "$PROFILE/predictions.jsonl" || true
printf '\nmetrics\n'
cat "$PROFILE/metrics.json" || true
printf '\nbenchmark\n'
cat "$PROFILE/benchmark.json" || true
printf '\ndebug files\n'
find "$PROFILE/debug" -maxdepth 2 -type f | sort || true

## Full 2-GPU Mini Training

Run these cells only after smoke passes and Kaggle is set to `GPU T4 x2`. Set `RUN_FULL_2GPU=True` in the first cell. Use `USE_SAFE_2GPU_CONFIG=True` if the default batch size OOMs.

In [ ]:
import os

if os.environ.get("USE_SAFE_2GPU_CONFIG") == "1":
    FULL_CONFIG = "configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml"
    FULL_PROFILE = "outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe"
else:
    FULL_CONFIG = "configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml"
    FULL_PROFILE = "outputs/repvit_t5_efficient_mini_kaggle_2gpu"

print({"full_config": FULL_CONFIG, "full_profile": FULL_PROFILE, "run_full_2gpu": os.environ.get("RUN_FULL_2GPU")})


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$RUN_FULL_2GPU" != "1" ]; then
  echo "Skipping full 2-GPU prepare-data/prepare-features. Set RUN_FULL_2GPU=True in the first cell."
  exit 0
fi
if [ "$USE_SAFE_2GPU_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
fi
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --debug --debug-samples 1
python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset full --debug --debug-samples 1


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$RUN_FULL_2GPU" != "1" ]; then
  echo "Skipping full 2-GPU align. Set RUN_FULL_2GPU=True in the first cell."
  exit 0
fi
if [ "$USE_SAFE_2GPU_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
fi
DBG=""
if [ "$FULL_DEBUG" = "1" ]; then DBG="--debug --debug-samples 1"; fi
accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align $DBG


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$RUN_FULL_2GPU" != "1" ]; then
  echo "Skipping full 2-GPU finetune. Set RUN_FULL_2GPU=True in the first cell."
  exit 0
fi
if [ "$USE_SAFE_2GPU_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu
fi
DBG=""
if [ "$FULL_DEBUG" = "1" ]; then DBG="--debug --debug-samples 1"; fi
accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume "$PROFILE/checkpoints/align_latest.pt" $DBG


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$RUN_FULL_2GPU" != "1" ]; then
  echo "Skipping full evaluate/benchmark. Set RUN_FULL_2GPU=True in the first cell."
  exit 0
fi
if [ "$USE_SAFE_2GPU_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu
fi
python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint "$PROFILE/checkpoints/finetune_latest.pt" --max-samples 1024 --debug --debug-samples 3
python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint "$PROFILE/checkpoints/finetune_latest.pt" --max-samples 128 --debug --debug-samples 3
